# Model Verification

This notebook collects some experiments used to verify the validity of the default
[xvamp.model.Duan2010](../xvamp/model.rst#xvamp.model.Duan2010) model.

> **Note:** There are a couple of options that can modify the default
> [xvamp.model.Duan2010](../xvamp/model.rst#xvamp.model.Duan2010) behavior, which are
> documented in the API, but not shown here.

## Venus Brightness Temperature

We start by initializing our default model:

In [ ]:
# some basic imports
import numpy as np
import matplotlib.pyplot as plt
from cmcrameri import cm
from astropy.units import Quantity
from astropy.visualization import quantity_support

# import reference model
from xvamp.model import Duan2010

# instantiate
model = Duan2010()

# initialize pretty plotting
quantity_support()
%config InlineBackend.figure_formats = ["svg", "pdf"]

The brightness temperature calculation requires the knowledge of the brightness
temperature at the surface, which is not something our model can compute.
However, we know the temperature of the atmosphere at the surface, and the surface
should be somewhat equilibrated with that temperature. So, we simply test our
model using a range of surface brightness temperatures that are derived from the
atmosphere's temperature through the surface emissivity:

In [ ]:
# import brightness computation function
from xvamp.utils import get_brightness_temperature

# define range of emissivities to apply to atmosphere temperature
# to derive surface brightness temperature
emissivity = np.linspace(0.5, 1.0, num=51)

# select some altitude levels for which to compute everything
ix_alt = np.arange(1, 15, 2)

# define range of surface brightness temperatures to assume
T_B_surface = emissivity[:, None] * model.temperature[ix_alt][None, :]

# for each assumed element in T_B_surface, get the total brightness temperature
T_B = Quantity(
    [
        get_brightness_temperature(
            model.altitude[i:],
            model.temperature[i:],
            model.refraction[i:],
            model.absorption[i:],
            Quantity(30, "°"),
            T_B_surface[:, ii],
        )
        for ii, i in enumerate(ix_alt)
    ]
)

In [ ]:
# import plotting functionality
from matplotlib.colors import Normalize

# start with the Ho & Kaufman ranges for X-band
plt.axhspan(Quantity(578, "K"), Quantity(657, "K"), fc="k", alpha=0.1, ec=None)
plt.axhspan(Quantity(500, "K"), Quantity(660, "K"), fc="k", alpha=0.1, ec=None)
# add our derived curves, colored by terrain altitude
norm = Normalize(
    vmin=model.altitude[ix_alt[0]].value, vmax=model.altitude[ix_alt[-1]].value
)
for ii, i in enumerate(ix_alt):
    plt.plot(
        emissivity,
        T_B[ii, :],
        c=cm.batlow(norm(model.altitude[i].value)),
        lw=2,
        label=f"{model.altitude[i]:.0f}",
    )
# make pretty
plt.xlabel("Surface Emissivity [-]")
plt.ylabel("Brightness Temperature [K]")
plt.legend(ncol=2)
plt.xlim(emissivity[0], emissivity[-1])
plt.ylim(400, 800)

## Model Intercomparison

In [ ]:
# standard imports
import numpy as np
import matplotlib.pyplot as plt
import astropy.units as u
from astropy.units import Quantity
from cmcrameri import cm

# import the default XVAMP model
from xvamp.model import Duan2010

# import the Magellan radio occultation results as well as the Stratton (1968)
# analytic refraction model
from xvamp.reference import magellan321x, stratton1968

### Refraction

For this section, we will follow the verification settings in the reference code,
which are designed to offer as close to a comparison with the Magellan data as
possible. To distinguish between the default model and the non-default model derived
for this verification section, we name them "Defaults" and "Verification", respectively.
For "Verification", we diverge from the default temperature, H2O, CO, SO2, H2SO4, and
OCS profiles, and select simpler ones. In particular, the H2SO4 profile is not the
one derived for global use, but one derived from the Magellan data itself.
We also neglect the influence of the clouds.

The Stratton (1968) analytic model is then evaluated on "Verification"'s profiles,
namely the temperature profile as well as the partial pressures of CO2 and H2O.
The Magellan experiment results are simply loaded. Finally, both the Stratton and
Magellan values are converted to the real part of the relative permittivity
for comparison with the XVAMP model.

In [ ]:
# get all-default model
m = Duan2010()
# initialize non-default XVAMP model
m_3212_ks = Duan2010(
    use_tpd_from="seiff:75",
    use_simple_h2o=True,
    use_simple_co=True,
    use_simple_so2=True,
    use_h2so4_from="kolodner:3212",
    use_ocs_from="simple",
    use_clouds_from="none",
)
# load Magellan data
mgn_abs, mgn_rtpd = magellan321x.get_wavelength_orbit("X", 3212)
# convert mean and plus/minus 3 s.d.
mgn_epsprimer = mgn_rtpd["REFRACTIVITY"].to(u.dimensionless_unscaled) ** 2
mgn_epsprimer_lower = (mgn_rtpd["REFRACTIVITY"] - 3 * mgn_rtpd["REFRACT_DEV"]).to(
    u.dimensionless_unscaled
).value ** 2
mgn_epsprimer_upper = (mgn_rtpd["REFRACTIVITY"] + 3 * mgn_rtpd["REFRACT_DEV"]).to(
    u.dimensionless_unscaled
).value ** 2
# evaluate Stratton model
stratton_epsprimer = (
    stratton1968(
        m_3212_ks.temperature,
        m_3212_ks.pressure * m_3212_ks.molar_fractions["CO2"],
        m_3212_ks.pressure * m_3212_ks.molar_fractions["N2"],
        m_3212_ks.pressure * m_3212_ks.molar_fractions["H2O"],
    ).to(u.dimensionless_unscaled)
    ** 2
)

In [ ]:
plt.plot(
    m.relative_permittivity.real,
    m.altitude,
    label="Defaults",
)
plt.plot(
    m_3212_ks.relative_permittivity.real,
    m_3212_ks.altitude,
    label="Verification",
)
plt.plot(
    mgn_epsprimer,
    mgn_rtpd["ALTITUDE"],
    ls="--",
    label="Magellan orbit 3212 ± 3σ",
)
plt.fill_betweenx(
    mgn_rtpd["ALTITUDE"].to("km").value,
    mgn_epsprimer_lower,
    mgn_epsprimer_upper,
    fc="C1",
    alpha=0.2,
)
plt.plot(
    stratton_epsprimer,
    m_3212_ks.altitude,
    ls=":",
    label="Stratton (1968)",
)
plt.xlabel("Real Part of the Relative Permittivity [-]")
plt.xlim(1, 1.004)
plt.ylabel("Altitude [km]")
plt.ylim(34, 98)
plt.legend()

As we can see, the XVAMP profiles closely match both the Stratton model and the Magellan
dataset. At this scale, for the real part of the relative permittivity, the model
settings "Defaults" and "Verification" do not yield meaningful differences.

### Absorptivity

In [ ]:
# get multiple profiles for comparison of the H2SO4 impact
m_default = Duan2010(
    use_tpd_from="seiff:75",
    use_simple_h2o=True,
    use_simple_co=True,
    use_simple_so2=True,
    use_h2so4_from="duan",
    use_ocs_from="simple",
    use_clouds_from="none",
)
m_3212_raw = Duan2010(
    use_tpd_from="seiff:75",
    use_simple_h2o=True,
    use_simple_co=True,
    use_simple_so2=True,
    use_h2so4_from="orbit:3212",
    use_ocs_from="simple",
    use_clouds_from="none",
)

We now have all the different validation test cases, and plot their absorptivity
profiles. We also add a plot of the part of the absorptivity that comes from the H2SO4
constituent, as it is one of the biggest drivers and will allow us to investigate the
results better.

In [ ]:
plt.figure(figsize=(8, 5), layout="constrained")
plt.plot(
    mgn_abs["ABSORPTIVITY"],
    mgn_abs["ALTITUDE"],
    c="k",
    label="Measured Magellan orbit 3212 ± 3σ",
)
plt.fill_betweenx(
    mgn_abs["ALTITUDE"].to("km").value,
    (mgn_abs["ABSORPTIVITY"] - 3 * mgn_abs["ABSORP_DEV"]).to_value("dB/km"),
    (mgn_abs["ABSORPTIVITY"] + 3 * mgn_abs["ABSORP_DEV"]).to_value("dB/km"),
    fc="k",
    alpha=0.2,
)
for i, (mdl, mdl_name) in enumerate(
    zip(
        [m, m_default, m_3212_ks, m_3212_raw],
        [
            "Defaults (H2SO4 from D10)",
            "Verification (H2SO4 from D10)",
            "Verification (H2SO4 from KS98)",
            "Verification (H2SO4 from J96)",
        ],
    )
):
    plt.plot(
        mdl.absorption.to_value("dB/km"),
        mdl.altitude,
        c=cm.batlowS(i + 1),
        label=mdl_name,
    )
    plt.plot(
        mdl.absorptions["H2SO4"].to_value("dB/km"),
        mdl.altitude,
        c=cm.batlowS(i + 1),
        ls="--",
    )
plt.grid()
plt.xlabel("Absorptivity [dB/km]")
plt.xlim(-0.02, 0.06)
plt.ylabel("Altitude [km]")
plt.ylim(35, 60)
plt.legend()
plt.title(
    "Total (solid) and H2SO4 contribution (dashed) for four different\n"
    "test cases compared with orbit 3212 Magellan measurements"
)

This plot is busy, so let's break it down. In black is the measured Magellan orbit 3212
absorptivity profile including the reported uncertainty (plotted is three times the
standard deviation). Ideally, our XVAMP model(s) will fall into the grey area.

The two validation test cases that use the D10 H2SO4 profiles decently fit the data
above 45 km altitude, but underestimate the absorptivity below 45 km.
For the two cases that use H2SO4 profiles derived from the experiment data itself
(J96 and KS98), we find that KS98 indeed does a good job, and only scratches the
three-standard-deviations boundary below 45 km altitude. The orbit 3212 profile
from J96 contains much more H2SO4, and yields a vastly higher absorptivity profile.

To summarize, we would therefore argue that our model, for the test case that most
closely resembles the conditions measured / expected for Magellan's orbit 3212
(namely the KS98 profile), reproduces to a satisfying degree the absorptivity profile
reported by the radio occultation experiment.